In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
import joblib

In [ ]:
search_paths = glob.glob("/kaggle/input/**/krishijal_master_dataset.csv", recursive=True) + glob.glob("**/krishijal_master_dataset.csv", recursive=True) + ["krishijal_master_dataset.csv"]
dataset_path = search_paths[0]
df = pd.read_csv(dataset_path)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(subset=["temperature", "rainfall", "humidity", "effective_soil_wetness", "crop_type", "growth_stage"])
df = df.drop_duplicates()

In [ ]:
selected_features = [
    "temperature", "temp_max", "temp_min", "rainfall", "humidity",
    "effective_soil_wetness", "solar_radiation", "soil_ph", "soil_organic_carbon",
    "ndvi", "et0", "etc", "crop_type", "growth_stage"
]
df_train = df[df["granularity"].isin(["daily", "monthly"])].copy()
X = df_train[selected_features].copy()
y_class = df_train["stress_level_code"].copy()
y_reg = df_train["irrigation_priority_index"].copy()

In [ ]:
crop_encoder = LabelEncoder()
X["crop_code"] = crop_encoder.fit_transform(X["crop_type"])

stage_encoder = LabelEncoder()
X["stage_code"] = stage_encoder.fit_transform(X["growth_stage"])

feature_cols = [
    "temperature", "temp_max", "temp_min", "rainfall", "humidity",
    "effective_soil_wetness", "solar_radiation", "soil_ph", "soil_organic_carbon",
    "ndvi", "et0", "etc", "crop_code", "stage_code"
]
X = X[feature_cols]

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(x=df_train["stress_level"], hue=df_train["stress_level"], order=["Low", "Moderate", "High", "Severe"], palette="viridis", legend=False)
plt.title("Distribution of Crop Stress Severity")
plt.xlabel("Stress Level")
plt.ylabel("Number of Observations")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df_train["irrigation_priority_index"], bins=25, kde=True, color="teal")
plt.title("Distribution of Irrigation Priority Index")
plt.xlabel("Irrigation Priority Index (0 to 100)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
sns.heatmap(X.corr(), annot=False, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(x=df_train["stress_level"], y=df_train["temperature"], hue=df_train["stress_level"], order=["Low", "Moderate", "High", "Severe"], palette="Set2", legend=False)
plt.title("Temperature by Crop Stress Severity")
plt.xlabel("Stress Level")
plt.ylabel("Temperature (°C)")
plt.show()

In [ ]:
X_train, X_test, y_train_class, y_test_class, y_train_reg, y_test_reg = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42, stratify=y_class
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols)

In [ ]:
model_logreg = LogisticRegression(max_iter=1000, random_state=42)
model_logreg.fit(X_train_scaled, y_train_class)

model_dt_class = DecisionTreeClassifier(max_depth=8, random_state=42)
model_dt_class.fit(X_train_scaled, y_train_class)

model_rf_class = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)
model_rf_class.fit(X_train_scaled, y_train_class)

In [ ]:
pred_logreg = model_logreg.predict(X_test_scaled)
pred_dt_class = model_dt_class.predict(X_test_scaled)
pred_rf_class = model_rf_class.predict(X_test_scaled)

classification_comparison = pd.DataFrame({
    "Algorithm": ["Logistic Regression", "Decision Tree Classifier", "Random Forest Classifier"],
    "Accuracy": [
        accuracy_score(y_test_class, pred_logreg),
        accuracy_score(y_test_class, pred_dt_class),
        accuracy_score(y_test_class, pred_rf_class)
    ],
    "Precision": [
        precision_score(y_test_class, pred_logreg, average="weighted", zero_division=0),
        precision_score(y_test_class, pred_dt_class, average="weighted", zero_division=0),
        precision_score(y_test_class, pred_rf_class, average="weighted", zero_division=0)
    ],
    "Recall": [
        recall_score(y_test_class, pred_logreg, average="weighted", zero_division=0),
        recall_score(y_test_class, pred_dt_class, average="weighted", zero_division=0),
        recall_score(y_test_class, pred_rf_class, average="weighted", zero_division=0)
    ],
    "F1-Score": [
        f1_score(y_test_class, pred_logreg, average="weighted", zero_division=0),
        f1_score(y_test_class, pred_dt_class, average="weighted", zero_division=0),
        f1_score(y_test_class, pred_rf_class, average="weighted", zero_division=0)
    ]
})
classification_comparison

In [ ]:
cm = confusion_matrix(y_test_class, pred_rf_class)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Low", "Mod", "High", "Sev"], yticklabels=["Low", "Mod", "High", "Sev"])
plt.title("Confusion Matrix — Random Forest Classifier")
plt.xlabel("Predicted Stress Level")
plt.ylabel("True Stress Level")
plt.show()

In [ ]:
model_linreg = LinearRegression()
model_linreg.fit(X_train_scaled, y_train_reg)

model_dt_reg = DecisionTreeRegressor(max_depth=8, random_state=42)
model_dt_reg.fit(X_train_scaled, y_train_reg)

model_rf_reg = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42)
model_rf_reg.fit(X_train_scaled, y_train_reg)

In [ ]:
pred_linreg = model_linreg.predict(X_test_scaled)
pred_dt_reg = model_dt_reg.predict(X_test_scaled)
pred_rf_reg = model_rf_reg.predict(X_test_scaled)

regression_comparison = pd.DataFrame({
    "Algorithm": ["Linear Regression", "Decision Tree Regressor", "Random Forest Regressor"],
    "MAE": [
        mean_absolute_error(y_test_reg, pred_linreg),
        mean_absolute_error(y_test_reg, pred_dt_reg),
        mean_absolute_error(y_test_reg, pred_rf_reg)
    ],
    "MSE": [
        mean_squared_error(y_test_reg, pred_linreg),
        mean_squared_error(y_test_reg, pred_dt_reg),
        mean_squared_error(y_test_reg, pred_rf_reg)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test_reg, pred_linreg)),
        np.sqrt(mean_squared_error(y_test_reg, pred_dt_reg)),
        np.sqrt(mean_squared_error(y_test_reg, pred_rf_reg))
    ],
    "R2 Score": [
        r2_score(y_test_reg, pred_linreg),
        r2_score(y_test_reg, pred_dt_reg),
        r2_score(y_test_reg, pred_rf_reg)
    ]
})
regression_comparison

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test_reg, pred_rf_reg, alpha=0.4, color="forestgreen", s=20)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], color="red", linestyle="--")
plt.title("Actual vs Predicted Irrigation Priority Index")
plt.xlabel("Actual Priority Index")
plt.ylabel("Predicted Priority Index")
plt.show()

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [8, 12]
}
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train_class)
best_model = grid_search.best_estimator_
grid_search.best_params_

In [ ]:
joblib.dump(scaler, "feature_scaler.pkl")
joblib.dump(best_model, "crop_stress_classifier.pkl")
joblib.dump(model_rf_reg, "irrigation_priority_regressor.pkl")
joblib.dump(model_dt_class, "crop_stress_decision_tree.pkl")
joblib.dump(model_logreg, "crop_stress_logistic_regression.pkl")
joblib.dump(model_dt_reg, "irrigation_decision_tree.pkl")
joblib.dump(model_linreg, "irrigation_linear_regression.pkl")

In [ ]:
df_districts = df[df["granularity"] == "district_baseline"].copy().reset_index(drop=True)
df_districts.head()

In [ ]:
X_732 = df_districts[feature_cols].copy()
X_732_scaled = pd.DataFrame(scaler.transform(X_732), columns=feature_cols)

In [ ]:
df_districts["Irrigation_Priority_Index"] = np.round(np.clip(model_rf_reg.predict(X_732_scaled), 0.0, 100.0), 1)
df_districts["Stress_Level_Code"] = model_rf_class.predict(X_732_scaled)
stress_label_dict = {0: "Low", 1: "Moderate", 2: "High", 3: "Severe"}
df_districts["Crop_Water_Stress"] = df_districts["Stress_Level_Code"].map(stress_label_dict)

df_districts["Priority_Category"] = np.where(
    df_districts["Irrigation_Priority_Index"] >= 50.0, "High Priority",
    np.where(df_districts["Irrigation_Priority_Index"] >= 35.0, "Medium Priority", "Low Priority")
)

df_districts["Recommended_Water_Depth"] = np.where(
    df_districts["Irrigation_Priority_Index"] >= 50.0, "40–50 mm",
    np.where(df_districts["Irrigation_Priority_Index"] >= 35.0, "25–35 mm", "Routine / Nil")
)

df_ranked_732 = df_districts.sort_values(by="Irrigation_Priority_Index", ascending=False).reset_index(drop=True)
df_ranked_732["Rank"] = range(1, len(df_ranked_732) + 1)
df_ranked_732[["Rank", "district", "state", "Irrigation_Priority_Index", "Priority_Category", "Recommended_Water_Depth"]].head(10)

In [ ]:
plt.figure(figsize=(9, 5))
top10_districts = df_ranked_732.head(10)
sns.barplot(
    data=top10_districts,
    x="Irrigation_Priority_Index",
    y="district",
    hue="district",
    palette="Reds_r",
    legend=False
)
plt.title("Top 10 Indian Districts by Irrigation Priority")
plt.xlabel("Irrigation Priority Index (0 to 100)")
plt.ylabel("District")
plt.show()

In [ ]:
priority_summary = df_ranked_732["Priority_Category"].value_counts()
priority_summary

In [ ]:
clustering_features = [
    "temperature", "rainfall", "humidity", "effective_soil_wetness",
    "solar_radiation", "soil_ph", "soil_organic_carbon", "ndvi", "et0", "etc"
]
scaler_clustering = StandardScaler()
X_cluster_scaled = scaler_clustering.fit_transform(df_ranked_732[clustering_features])

kmeans_model = KMeans(n_clusters=4, random_state=42, n_init=10)
df_ranked_732["Cluster_ID"] = kmeans_model.fit_predict(X_cluster_scaled)

zone_names = {
    0: "Zone 0: Cool Mountain & High-Altitude",
    1: "Zone 1: Humid Coastal & Floodplain",
    2: "Zone 2: Semi-Arid & Deficit Core",
    3: "Zone 3: Wet Tropical & Heavy Rain"
}
df_ranked_732["Cluster_Zone"] = df_ranked_732["Cluster_ID"].map(zone_names)
df_ranked_732["Cluster_Zone"].value_counts()

In [ ]:
pca_model = PCA(n_components=2, random_state=42)
pca_coordinates = pca_model.fit_transform(X_cluster_scaled)
df_ranked_732["PCA_X"] = np.round(pca_coordinates[:, 0], 2)
df_ranked_732["PCA_Y"] = np.round(pca_coordinates[:, 1], 2)
pca_model.explained_variance_ratio_

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_ranked_732,
    x="PCA_X",
    y="PCA_Y",
    hue="Cluster_Zone",
    palette=["#3b82f6", "#10b981", "#f59e0b", "#ef4444"],
    alpha=0.8,
    s=45
)
plt.title("All-India 732 Districts Agro-Climatic PCA Projection")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
cluster_characterization = df_ranked_732.groupby("Cluster_Zone")[
    ["temperature", "rainfall", "humidity", "effective_soil_wetness", "Irrigation_Priority_Index"]
].mean().round(2)
cluster_characterization

In [ ]:
df_ranked_732.to_csv("all_india_732_districts_priority_and_clusters.csv", index=False)
df_ranked_732.shape